# Predicting Range Errors
<span style="font-family: 'Helvetica'; font-size: 15px;">
    Range measurements are also quantized in ISAC simulations, however its difficult to find a reliable rule like we did with angles. Instead, we will attempt to classify targets into several detection regimes. They are : bulk, sideband+, sideband-, far+, far-, and near. For more information, see the distributional pictures during evaluation.
</span>

### Classifying Regimes

<span style="font-family: 'Helvetica'; font-size: 15px;">
    To classify the regime, we use the RegimeClassifier class. Note that we can optionally input a validation set for early stopping. While this isn't required, it is highly recommended because when left to run for long times, we face the risk of heavily overfitting to the small amount of outliers. For this reason, we also use 5 fold cross validation, and then use the average of the 5 models. Note that here, we need to pass in the tags associated so that the model knows how to assign cross validation splits. Additionally, the model applies a bias to each logit that is predicted by a separate linear regression model.
</span>

In [2]:
import sys
import polars as pl
import numpy as np
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src.data.load import load_pos, group_split
from src.components.range.regime import RegimeClassifier, save_class_outputs, load_class_model

tags, labels, features = load_pos()
[X, X_val, y, y_val, t, t_val] = group_split(tags, labels, features, return_tags=True)

classifier = RegimeClassifier()
train_df = pl.concat([X, y, t], how = "horizontal")
val_df = pl.concat([X_val, y_val, t_val], how = "horizontal")

/tmp/ipykernel_259748/4178198705.py:13: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  train_df = pl.concat([X, y, t], how = "horizontal")
/tmp/ipykernel_259748/4178198705.py:14: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  val_df = pl.concat([X_val, y_val, t_val], how = "horizontal")


<span style="font-family: 'Helvetica'; font-size: 15px;">
    For testing, fit returns the raw logits, the probabilities and the bias values. Because training can be very long, the save_class_outputs function can be used to quickly cache training runs.
</span>

In [3]:
oof_logits, oof_cal, b_final, models = classifier.fit(train_df, val_df)
save_class_outputs(oof_logits, oof_cal, b_final, models, "models/test_range_1")

Cross fitting fold 0
Training until validation scores don't improve for 100 rounds
[50]	valid's multi_logloss: 0.104247
[100]	valid's multi_logloss: 0.103548
[150]	valid's multi_logloss: 0.106943
Early stopping, best iteration is:
[82]	valid's multi_logloss: 0.103229
82
Cross fitting fold 1
Training until validation scores don't improve for 100 rounds
[50]	valid's multi_logloss: 0.103704
[100]	valid's multi_logloss: 0.104429
[150]	valid's multi_logloss: 0.107541
Early stopping, best iteration is:
[64]	valid's multi_logloss: 0.103065
64
Cross fitting fold 2
Training until validation scores don't improve for 100 rounds
[50]	valid's multi_logloss: 0.105381
[100]	valid's multi_logloss: 0.105483
[150]	valid's multi_logloss: 0.108555
Early stopping, best iteration is:
[61]	valid's multi_logloss: 0.104785
61
Cross fitting fold 3
Training until validation scores don't improve for 100 rounds
[50]	valid's multi_logloss: 0.105845
[100]	valid's multi_logloss: 0.106363
[150]	valid's multi_logloss: 

<span style="font-family: 'Helvetica'; font-size: 15px;">
    Ideally, we should use a completely separate dataset at inference (but val is good enough for now). predict_proba returns the probabilities for each class per target, and use sample with a random number generator to get each class.
</span>

In [14]:
oof_logits, oof_cal, b_final, models = load_class_model("models/test_range_1")
rng = np.random.default_rng(seed = 42)
r_classifier = RegimeClassifier(models = models)
r_classifier.bias = b_final
probs = r_classifier.predict_proba(X_val)
classes = r_classifier.sample(probs, rng)
print(classes)

shape: (9_011, 2)
┌────────┬────────────┐
│ region ┆ region_txt │
│ ---    ┆ ---        │
│ i64    ┆ str        │
╞════════╪════════════╡
│ 0      ┆ bulk       │
│ 0      ┆ bulk       │
│ 0      ┆ bulk       │
│ 0      ┆ bulk       │
│ 0      ┆ bulk       │
│ …      ┆ …          │
│ 0      ┆ bulk       │
│ 0      ┆ bulk       │
│ 0      ┆ bulk       │
│ 0      ┆ bulk       │
│ 0      ┆ bulk       │
└────────┴────────────┘


### Predicting Bulk Errors

<span style="font-family: 'Helvetica'; font-size: 15px;">
    Now we partition the dataset into bulk samples and non-bulk samples. We've found that the bulk targets follow an almost uniform distribution with a strong correlation with respect to their distance from the antenna. Namely, plotting errors vs range fills out a rectangle of errors that is rotated. To recreate this distribution, we fit a linear regression model for the slope of the rectangle. Then at inference time, we predict the value of the average error due to the range, and then sample a random residual. To obtain this distribution of residuals, call BulkModel.fit on the training data
</span>

In [21]:
from src.components.range.distributions import label_regime, BulkModel

tags, labels, features = load_pos()
labels = label_regime(labels)
[X, X_val, y, y_val] = group_split(tags, labels, features)
b_model = BulkModel()

df = pl.concat([X, y], how = "horizontal")
df = df.filter(pl.col("region_txt") == "bulk")
X1 = df.select("bistatic_range_m")
y1 = df.select("range_error_m")
residuals = b_model.fit(X1, y1)


/tmp/ipykernel_259748/455393328.py:8: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  df = pl.concat([X, y], how = "horizontal")


<span style="font-family: 'Helvetica'; font-size: 15px;">
    Then, at inference time, simply pass in the feature data. The distributions is saved internally in the model
</span>

In [22]:
rng = np.random.default_rng(seed = 42)
samples = b_model.sample(X1, rng)
print(samples)

shape: (34_561, 1)
┌───────────────┐
│ sampled_error │
│ ---           │
│ f64           │
╞═══════════════╡
│ 0.223924      │
│ -0.295281     │
│ -0.299147     │
│ 0.283428      │
│ 0.036538      │
│ …             │
│ 0.639397      │
│ 0.495858      │
│ -0.271654     │
│ 0.567655      │
│ 0.250168      │
└───────────────┘


### Predicting Non-Bulk Errors

<span style="font-family: 'Helvetica'; font-size: 15px;">
    For non  samples, we simply sample for the constructed distribution according to regime, which is found during training. For very high errors falling under far+, we sample based off their nn_norm_range_sep quantile which seems to have correlation with how large to make the blunder. In addition, we add a jitter to the sampled error in order to get values that weren't seen in the training data. This matters for very rare classes in which we have few data points to sample from over a large region
</span>

In [23]:
from src.components.range.distributions import NonBulkModel


nb_model = NonBulkModel()
nb_model.fit(X, y)

# Predict classes from validation set
probs = r_classifier.predict_proba(X_val)
classes = r_classifier.sample(probs, rng)

# select features
df = pl.concat([X_val,classes], how = "horizontal")
df = df.filter(pl.col("region_txt") != "bulk")


# Sample from predicted classes 
nb_model.sample(df, rng)

/tmp/ipykernel_259748/3857827903.py:12: DeprecationWarning: the default behavior of `how='horizontal'` for `concat` is deprecated and will require equal heights in the next breaking release. Use `how='horizontal_extend'` to keep the current behavior.
(Deprecated in version 1.42.1)
  df = pl.concat([X_val,classes], how = "horizontal")


sampled_error
f64
1.636016
-1.993931
-2.084053
-1.701643
-1.780629
…
-4.217784
13.01565
1.621052


### Full Range Pipeline

<span style="font-family: 'Helvetica'; font-size: 15px;">
    The RangeModel class stitches together all three components into one single pipeline. The training data is classified and sampled by RegimeClassifier, partitioned into sampled bulk and non bulk targets, and then range error is sampled from the two sets before reassembling them back together.
</span>

In [28]:
from src.components.range.range import RangeModel

range_model = RangeModel()  
range_model.fit(train_df, val_df)
range_model.save("models/range_model.joblib")

Cross fitting fold 0
Training until validation scores don't improve for 100 rounds
[50]	valid's multi_logloss: 0.104248
[100]	valid's multi_logloss: 0.10355
[150]	valid's multi_logloss: 0.106945
Early stopping, best iteration is:
[82]	valid's multi_logloss: 0.103231
82
Cross fitting fold 1
Training until validation scores don't improve for 100 rounds
[50]	valid's multi_logloss: 0.103704
[100]	valid's multi_logloss: 0.10443
[150]	valid's multi_logloss: 0.107565
Early stopping, best iteration is:
[64]	valid's multi_logloss: 0.103065
64
Cross fitting fold 2
Training until validation scores don't improve for 100 rounds
[50]	valid's multi_logloss: 0.105383
[100]	valid's multi_logloss: 0.105489
[150]	valid's multi_logloss: 0.108561
Early stopping, best iteration is:
[61]	valid's multi_logloss: 0.104787
61
Cross fitting fold 3
Training until validation scores don't improve for 100 rounds
[50]	valid's multi_logloss: 0.105844
[100]	valid's multi_logloss: 0.106362
[150]	valid's multi_logloss: 0.

In [32]:
range_model = RangeModel.load("models/range_model.joblib")
probs = range_model.predict_proba(X_val)
samples = range_model.sample(X_val, probs, rng)
print(samples)

shape: (9_011, 1)
┌───────────────┐
│ sampled_error │
│ ---           │
│ f64           │
╞═══════════════╡
│ -0.206466     │
│ 0.235945      │
│ 0.238085      │
│ 0.933966      │
│ 0.106566      │
│ …             │
│ 0.246513      │
│ 0.417087      │
│ 0.388619      │
│ -0.214929     │
│ 0.106495      │
└───────────────┘
